### The Core Idea
This dataset is **synthetically generated**, which means the target label
(`Irrigation_Need`) was programmatically assigned based on a set of 
**threshold rules** applied to the raw features.

By analysing the original dataset, we can reverse engineer these rules
and build a single composite score that almost perfectly predicts the target
**before any model is even trained.** You can check the details [here](https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687460) in a discussion post by **CHRIS DEOTTE**. Here we'll try to engineer thoes features and see if our model performs better with this extra bit of information.

## Importing required Modules and Libraries

In [28]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd 
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

## Importing Data:

In [29]:
path = '/kaggle/input/competitions/playground-series-s6e4/'
orig_path = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train_df = pd.read_csv(path + 'train.csv')
display(train_df.head())
print('Training set: ', train_df.shape)

orig_df = pd.read_csv(orig_path)
display(orig_df.head())
print('Orignal set: ', orig_df.shape)

test_df = pd.read_csv(path + 'test.csv')
display(test_df.head())
print('Test set: ', test_df.shape)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Training set:  (630000, 21)


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


Orignal set:  (10000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


Test set:  (270000, 20)


In [30]:
target = 'Irrigation_Need'

X = train_df.drop(columns=['id', target], axis=1)
X_orig = orig_df.drop(columns=[target], axis=1)

y = train_df[target].map({'Low': 0, 'Medium': 1, 'High': 2})
y_orig = orig_df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

X_test = test_df.drop('id', axis=1)

In [31]:
combined_df = pd.concat([X, X_orig], ignore_index=True)
y_combined = pd.concat([y, y_orig], ignore_index=True)
combined_df.shape

(640000, 19)

## Feature Engineering:

In [32]:
num_cols = combined_df.select_dtypes(include='number').columns
cat_cols = combined_df.select_dtypes("object").columns

In [33]:
#thresholds for num features
Soil_Moisture_thresh = 25
Rainfall_mm_thresh = 300
Temperature_C_thresh = 30
Wind_Speed_kmh_thresh = 10

#categories for cat features
Crop_Growth_Stage_thresh = 'Harvest'
Crop_Growth_Stage_thresh = 'Sowing'
Mulching_Used_thresh = 'Yes'

In [34]:
def binary_magic_features(df):
    
    df["soil_25"]      = (df["Soil_Moisture"]       < Soil_Moisture_thresh).astype(int)
    df["rain_300"]     = (df["Rainfall_mm"]         < Rainfall_mm_thresh).astype(int)
    df["temp_30"]      = (df["Temperature_C"]       > Temperature_C_thresh).astype(int)
    df["wind_10"]      = (df["Wind_Speed_kmh"]      > Wind_Speed_kmh_thresh).astype(int)  
    df["is_harvest"]   = (df["Crop_Growth_Stage"]  == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"]  == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]      == "Yes").astype(int)
    return df


def magic_score(df):
    
    high = 2 * df['soil_25'] + 2 * df['rain_300'] + df['temp_30'] + df['wind_10']
    low = 2 * df['is_harvest'] + 2 * df['is_sowing'] + df['mulching_yes']
    df['magic_score'] = high - low
    return df


def create_features(df):
    df = binary_magic_features(df)
    df = magic_score(df)
    return df

In [35]:
combined_df = create_features(combined_df)
X_test = create_features(X_test)

### Feature Crosses:
Writting a custom transformer just to keep it clean

In [36]:
def feature_crosses(X):
        X = X.copy()

        # Convert to string first
        cat_cols = X.select_dtypes(include=['category', 'object']).columns
        X[cat_cols] = X[cat_cols].astype(str)
        
        # Create _orig copies
        for col in cat_cols:
            X[f'{col}_orig'] = X[col]
        
        # Numerical crosses
        X['heat_humidity_stress'] = X['Temperature_C'] * (1 - X['Humidity']/100)
        X['water_availability']   = X['Rainfall_mm'] + X['Previous_Irrigation_mm']
        X['evaporation_proxy']    = X['Temperature_C'] * X['Wind_Speed_kmh'] * X['Sunlight_Hours']
        X['soil_health']          = X['Organic_Carbon'] * X['Soil_Moisture']

        #Cat x Cat
        X['region_season']     = X['Region'].astype(str) + '_' + X['Season'].astype(str)
        X['crop_growth']       = X['Crop_Type'].astype(str) + '_' + X['Crop_Growth_Stage'].astype(str)
        X['soil_crop']         = X['Soil_Type'].astype(str) + '_' + X['Crop_Type'].astype(str)
        X['source_irrigation'] = X['Water_Source'].astype(str) + '_' + X['Irrigation_Type'].astype(str)
        
        # Season cyclical
        season_map = {'Spring':0, 'Summer':1, 'Autumn':2, 'Winter':3}
        X['season_num'] = X['Season'].astype(str).map(season_map)
        X['season_sin'] = np.sin(2 * np.pi * X['season_num'] / 4)
        X['season_cos'] = np.cos(2 * np.pi * X['season_num'] / 4)
        X.drop('season_num', axis=1, inplace=True)

        #Domain Specific
        X['water_stress'] = (X['Temperature_C'] * X['Wind_Speed_kmh']) / (X['Rainfall_mm'] + X['Soil_Moisture'] + 1) # Water stress index 
        X['crop_demand']  = X['Temperature_C'] * X['Sunlight_Hours'] * (1 - X['Humidity']/100) # Crop water demand proxy
        X['retention']    = X['Soil_Moisture'] * X['Organic_Carbon'] * X['Electrical_Conductivity'] # Soil water retention
        X['irr_efficiency'] = X['Previous_Irrigation_mm'] / (X['Field_Area_hectare'] + 1) # Irrigation efficiency
        X['rain_per_area'] = X['Rainfall_mm'] / (X['Field_Area_hectare'] + 1)
    
        return X

## OOF predictions With scaling and OHE(one hot encoding):
Wont be doing it in the cleanest way possible in a little time crunch, so overlook it please

In [37]:
# num_cols = X.select_dtypes(include=['number']).columns
# cat_cols = X.select_dtypes(include=['object']).columns

In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [39]:
X_processed = preprocessor.fit_transform(combined_df)
X_test_processed = preprocessor.transform(X_test)

In [40]:
xgb_params = {
    'learning_rate': 0.03, 
    'max_depth': 4, 
    'min_child_weight': 2,
    'subsample': 0.98,
    'gamma': 4.2, 
    'colsample_bytree': 0.5, 
    'lambda': 0.001,
    'alpha': 4.1,
    'n_estimators': 50_000, 
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'device': 'cuda',
    'num_class': 3
}

In [42]:
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)
bas_scores = []
n_classes = y.nunique()
oof_test = np.zeros((X_test_processed.shape[0], n_classes))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_processed, y_combined)):
    X_tr, X_val = X_processed[tr_idx], X_processed[val_idx]
    y_tr, y_val = y_combined.iloc[tr_idx], y_combined.iloc[val_idx]

    sw = compute_sample_weight(class_weight="balanced", y=y_tr)
    
    model = XGBClassifier(
        **xgb_params,
        random_state=42,
        
        callbacks=[
            xgb.callback.EarlyStopping( #early stopping callback
                rounds=200,
                save_best=True,
            )
        ]
    )

    model.fit(
        X_tr,
        y_tr,
        sample_weight=sw,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)

    test_preds = model.predict_proba(X_test_processed)
    oof_test += test_preds / skf.n_splits

    bas = balanced_accuracy_score(y_val, preds)
    bas_scores.append(bas)
    print(f"Fold {fold} BAS: {bas}")
print(f"Mean BAS: {np.mean(bas_scores)}")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:57:28] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 0 BAS: 0.9727263380649509
Fold 1 BAS: 0.9718374904560522
Fold 2 BAS: 0.9737700583112171
Fold 3 BAS: 0.9724395241022523
Fold 4 BAS: 0.9733471662294737
Fold 5 BAS: 0.9717601729457656
Fold 6 BAS: 0.9732874855579778
Fold 7 BAS: 0.9689907274396211
Mean BAS: 0.9722698703884138


In [43]:
preds = oof_test.argmax(axis=1)
preds = pd.Series(preds).map({0: "Low", 1: "Medium", 2: "High"})

submission = pd.DataFrame({
    "id": test_df["id"],
    target: preds
})

submission.to_csv("submission_magic_score.csv", index=False)